# **Recency , Frequency, Monetary (RFM) Customer Segmentation**

## 1\. customer with Sales Recency, Frequency, Monetary

In [4]:
DROP TABLE IF EXISTS #CUST_RAW
SELECT MAIN_IDENTIFIER_NO, LAST_DATE
        ,COALESCE(RECENCY,730) AS RECENCY
        ,COALESCE(FREQUENCY,0) AS FREQUENCY
        ,COALESCE(MONETARY,0) AS MONETARY
INTO #CUST_RAW
FROM customer AS CUST
LEFT JOIN (SELECT RWD_CARD_NO
            ,MAX(BILLING_DATE) AS LAST_DATE
            ,DATEDIFF(DAY,MAX(BILLING_DATE),'2025-06-30') AS RECENCY
            ,COUNT(DISTINCT BILLING_DATE) AS FREQUENCY
            ,SUM(NET_ITEM_AMT) AS MONETARY
            FROM billing
            WHERE  (BILLING_DATE BETWEEN DATEADD(Month,-24,'2025-06-30') AND '2025-06-30') AND RWD_CARD_NO IS NOT NULL
            GROUP BY RWD_CARD_NO) AS SALES_2Y
ON CUST.MAIN_IDENTIFIER_NO = RWD_CARD_NO
WHERE MAIN_IDENTIFIER_NO IS NOT NULL AND CUS_STATUS = 'A' AND RWD_CARD_NO IS NOT NULL AND MONETARY > 0

SELECT TOP 5*
FROM #CUST_RAW

(4247306 rows affected)

(5 rows affected)

Total execution time: 00:00:40.777

MAIN_IDENTIFIER_NO,LAST_DATE,RECENCY,FREQUENCY,MONETARY
2007642456,2025-06-30,0,1,1050
1054704520,2025-05-03,58,11,8864.82
2004541646,2025-04-12,79,2,13926
1020824034,2025-06-26,4,5,12907
1093086745,2024-04-08,448,1,24895.12


## 2\. RECENCY, FEQUENCY, MONETARY to Decile RFM

In [5]:
DROP TABLE IF EXISTS #CUST_QT
SELECT *
        ,NTILE(10) OVER(ORDER BY RECENCY DESC) AS R 
        ,NTILE(10) OVER(ORDER BY FREQUENCY ASC) AS F
        ,NTILE(10) OVER(ORDER BY MONETARY ASC) AS M
INTO #CUST_QT
FROM #CUST_RAW


SELECT TOP 5*
FROM #CUST_QT

(4247306 rows affected)

(5 rows affected)

Total execution time: 00:00:29.060

MAIN_IDENTIFIER_NO,LAST_DATE,RECENCY,FREQUENCY,MONETARY,R,F,M
2004642893,2024-12-24,188,3,1.1368683772161603E-13,5,5,1
1052274323,2024-12-04,208,1,1,4,2,1
1057594369,2024-02-26,490,1,1,2,1,1
2006791322,2025-02-27,123,1,2,6,3,1
2005825844,2025-04-27,64,1,2,7,3,1


## 3.  RFM segment , RFM Score

In [6]:
DROP TABLE IF EXISTS #CUST_RFM
SELECT *
        ,CONCAT(R,F,M) AS RFM_SEGMENT
        ,SUM(R+F+M) AS RFM_SCORE
INTO #CUST_RFM
FROM #CUST_QT
GROUP BY MAIN_IDENTIFIER_NO, LAST_DATE, RECENCY, FREQUENCY, MONETARY, R, F, M
ORDER BY SUM(R+F+M) DESC

SELECT TOP 10*
FROM #CUST_RFM

(4247306 rows affected)

(10 rows affected)

Total execution time: 00:00:02.986

MAIN_IDENTIFIER_NO,LAST_DATE,RECENCY,FREQUENCY,MONETARY,R,F,M,RFM_SEGMENT,RFM_SCORE
1002336309,2025-04-15,76,7,8175.6900000000005,7,8,5,785,20
1002336317,2025-06-01,29,10,40431.49,9,8,9,989,26
1002336325,2025-06-02,28,1,2558,9,3,3,933,15
1002336490,2025-05-07,54,3,41420,8,6,9,869,23
1002336694,2025-06-28,2,4,24506,10,6,8,1068,24
1002337160,2025-03-28,94,1,3390,6,3,4,634,13
1002337313,2025-04-14,77,13,18042,7,9,7,797,23
1002337381,2024-07-18,347,2,2842,3,4,3,343,10
1002337437,2024-11-09,233,1,4614,4,2,4,424,10
1002337593,2025-04-12,79,23,87879.31000000001,7,10,10,71010,27


## 4.Metrics per RFM score

In [7]:
DROP TABLE IF EXISTS #CUST_SCORE
SELECT RFM_SCORE
        ,AVG(RECENCY) AS RECENCY_MEAN
        ,AVG(FREQUENCY) AS FREQUENCY_MEAN
        ,AVG(MONETARY) AS MONETARY_MEAN
        ,COUNT(MONETARY) AS MONETARY_COUNT
INTO #CUST_SCORE
FROM #CUST_RFM
GROUP BY RFM_SCORE
ORDER BY RFM_SCORE

SELECT *
FROM #CUST_SCORE
ORDER BY RFM_SCORE

(28 rows affected)

(28 rows affected)

Total execution time: 00:00:00.158

RFM_SCORE,RECENCY_MEAN,FREQUENCY_MEAN,MONETARY_MEAN,MONETARY_COUNT
3,640,1,366.34139248250034,99714
4,585,1,898.4685992744501,90139
5,543,1,1525.7514790452653,89240
6,460,1,1812.829075883739,112053
7,422,1,2344.650757061021,138932
8,428,1,3426.7516098907217,139525
9,386,1,4245.714155055611,163709
10,344,1,4999.009795143241,177197
11,302,1,5543.543056461174,189245
12,265,2,6134.075142886444,194735


## 5.Name Segment

In [8]:
SELECT SCORE.RFM_SCORE
        ,CASE WHEN SCORE.RFM_SCORE <= 30 AND SCORE.RFM_SCORE >= 25 THEN 'Platinum'
            WHEN SCORE.RFM_SCORE < 25 AND SCORE.RFM_SCORE >= 19 THEN 'Gold'
            WHEN SCORE.RFM_SCORE < 19 AND SCORE.RFM_SCORE >= 13 THEN 'Silver'
            WHEN SCORE.RFM_SCORE < 13 AND SCORE.RFM_SCORE >= 7 THEN 'Bronze'
            WHEN SCORE.RFM_SCORE < 7 AND SCORE.RFM_SCORE >= 3 THEN 'Churn_Risk'
            END AS SEGMENT_NAME
        ,AVG(RECENCY) AS RECENCY_MEAN
        ,AVG(FREQUENCY) AS FREQUENCY_MEAN
        ,AVG(MONETARY) AS MONETARY_MEAN
        ,COUNT(MONETARY) AS MONETARY_COUNT
FROM #CUST_SCORE AS SCORE
LEFT JOIN (SELECT * FROM #CUST_RFM) AS RFM
ON SCORE.RFM_SCORE = RFM.RFM_SCORE
GROUP BY CASE WHEN SCORE.RFM_SCORE <= 30 AND SCORE.RFM_SCORE >= 25 THEN 'Platinum'
            WHEN SCORE.RFM_SCORE < 25 AND SCORE.RFM_SCORE >= 19 THEN 'Gold'
            WHEN SCORE.RFM_SCORE < 19 AND SCORE.RFM_SCORE >= 13 THEN 'Silver'
            WHEN SCORE.RFM_SCORE < 13 AND SCORE.RFM_SCORE >= 7 THEN 'Bronze'
            WHEN SCORE.RFM_SCORE < 7 AND SCORE.RFM_SCORE >= 3 THEN 'Churn_Risk'
            END
        ,SCORE.RFM_SCORE
ORDER BY SCORE.RFM_SCORE DESC

(28 rows affected)

Total execution time: 00:00:00.394

RFM_SCORE,SEGMENT_NAME,RECENCY_MEAN,FREQUENCY_MEAN,MONETARY_MEAN,MONETARY_COUNT
30,Platinum,4,55,293989.96882894775,95701
29,Platinum,13,32,145995.16188318637,105162
28,Platinum,22,24,104058.69875847112,121278
27,Platinum,31,19,77761.83698573637,129607
26,Platinum,41,16,62797.244673551264,138809
25,Platinum,51,13,51589.43393907247,146649
24,Gold,63,11,42655.69237013173,153456
23,Gold,75,9,35472.64893721916,158508
22,Gold,89,8,30049.33645159739,160272
21,Gold,102,7,25745.907606488803,163588
